# Agente de IA con LangChain

Este notebook contiene un agente de IA conversacional usando LangChain y Groq, con interfaz gráfica tipo ChatGPT.

In [1]:
# Instalación de dependencias necesarias
# Ejecuta esta celda solo la primera vez o si necesitas instalar las librerías
# Nota: langchain-classic es necesario para ConversationChain y ConversationBufferMemory

%pip install --upgrade langchain langchain-classic langchain-community langchain-core langchain-groq gradio python-dotenv reportlab python-docx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importar las librerías necesarias
import os

# Importar dotenv para gestión de variables de entorno
# Nota: Si el IDE muestra error, ignóralo - el paquete está instalado y funcionará en ejecución
try:
    from dotenv import load_dotenv  # type: ignore
except ImportError:
    # Si falla la importación, instalar el paquete
    import subprocess
    import sys
    print("Instalando python-dotenv...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "--quiet"])
    from dotenv import load_dotenv  # type: ignore

# Importar ChatGroq de langchain_groq
# Nota: Si el IDE muestra error, ignóralo - el paquete está instalado y funcionará en ejecución
try:
    from langchain_groq import ChatGroq  # type: ignore
except ImportError:
    # Si falla la importación, instalar el paquete
    import subprocess
    import sys
    print("Instalando langchain-groq...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-groq", "--quiet"])
    from langchain_groq import ChatGroq  # type: ignore

# Importar prompts de langchain_core
# Nota: Si el IDE muestra error, ignóralo - el paquete está instalado y funcionará en ejecución
from langchain_core.prompts import (  # type: ignore
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
# Importar chains y memory de langchain-classic
# En LangChain 1.2+, estos módulos se movieron a langchain-classic
try:
    from langchain_classic.chains import ConversationChain  # type: ignore
    from langchain_classic.memory import ConversationBufferMemory  # type: ignore
except ImportError:
    # Fallback: intentar importar desde langchain (versiones antiguas)
    try:
        from langchain.chains import ConversationChain  # type: ignore
        from langchain.memory import ConversationBufferMemory  # type: ignore
    except ImportError:
        # Si falla, instalar langchain-classic
        import subprocess
        import sys
        print("Instalando langchain-classic...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-classic", "--quiet"])
        from langchain_classic.chains import ConversationChain  # type: ignore
        from langchain_classic.memory import ConversationBufferMemory  # type: ignore
# Importar gradio para la interfaz gráfica
# Nota: Si el IDE muestra error, ignóralo - el paquete está instalado y funcionará en ejecución
import gradio as gr  # type: ignore
from typing import Tuple, Optional
from datetime import datetime
import tempfile

# Importar librerías para exportar conversaciones
# Nota: Si el IDE muestra error, ignóralo - el paquete está instalado y funcionará en ejecución
try:
    from reportlab.lib.pagesizes import letter  # type: ignore
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle  # type: ignore
    from reportlab.lib.units import inch  # type: ignore
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer  # type: ignore
    from reportlab.lib.enums import TA_LEFT  # type: ignore
    from reportlab.lib.colors import HexColor  # type: ignore
except ImportError:
    print("Instalando reportlab...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab", "--quiet"])
    from reportlab.lib.pagesizes import letter  # type: ignore
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle  # type: ignore
    from reportlab.lib.units import inch  # type: ignore
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer  # type: ignore
    from reportlab.lib.enums import TA_LEFT  # type: ignore
    from reportlab.lib.colors import HexColor  # type: ignore

try:
    from docx import Document  # type: ignore
    from docx.shared import Pt, RGBColor  # type: ignore
    from docx.enum.text import WD_ALIGN_PARAGRAPH  # type: ignore
except ImportError:
    print("Instalando python-docx...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx", "--quiet"])
    from docx import Document  # type: ignore
    from docx.shared import Pt, RGBColor  # type: ignore
    from docx.enum.text import WD_ALIGN_PARAGRAPH  # type: ignore

In [3]:
# Cargar variables de entorno desde el archivo .env
# Esto carga automáticamente todas las variables definidas en .env
load_dotenv()

# Obtener la API Key de Groq desde las variables de entorno
api_key = os.getenv("GROQ_API_KEY")

if api_key:
    # Asegurar que la variable de entorno esté configurada
    os.environ["GROQ_API_KEY"] = api_key
    print("✓ API Key cargada correctamente desde .env")
else:
    print("✗ Error: No se encontró GROQ_API_KEY en el archivo .env")
    print("  Por favor, crea un archivo .env con la siguiente línea:")
    print("  GROQ_API_KEY=tu_api_key_aqui")

✓ API Key cargada correctamente desde .env


In [4]:
# Inicializar el modelo de Groq y la memoria de conversación
def initialize_agent():
    """Inicializa el agente de IA con memoria de conversación"""
    
    # Inicializar el modelo de Groq (puedes cambiar el modelo según tus necesidades)
    # Modelos disponibles actualmente (2024):
    # - 'llama-3.1-8b-instant': Modelo rápido y eficiente (recomendado)
    # - 'llama-3.3-70b-versatile': Modelo potente (si está disponible)
    # - 'mixtral-8x7b-32768': Modelo con contexto largo
    # - 'llama-3.1-70b-versatile': DESCONTINUADO - no usar
    llm = ChatGroq(
        groq_api_key=os.environ.get("GROQ_API_KEY"),
        model_name="llama-3.1-8b-instant",  # Modelo rápido y disponible
        temperature=0.7,  # Controla la creatividad (0.0-1.0)
        max_tokens=2048
    )
    
    # Crear memoria para mantener el contexto de la conversación
    memory = ConversationBufferMemory(
        return_messages=True,
        memory_key="history"
    )
    
    # Crear el prompt template para la conversación
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            "Eres un asistente de IA útil, amigable y conversacional. "
            "Responde de manera clara y natural, manteniendo el contexto de la conversación. "
            "Si no sabes algo, admítelo honestamente."
        ),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template("{input}")
    ])
    
    # Crear la cadena de conversación
    # NOTA: ConversationChain está deprecated en versiones recientes de LangChain
    # pero sigue funcionando correctamente. Para futuras versiones, considera migrar a LCEL (LangChain Expression Language)
    conversation = ConversationChain(
        llm=llm,
        memory=memory,
        prompt=prompt,
        verbose=False
    )
    
    return conversation

# Inicializar el agente
conversation_chain = initialize_agent()
print("✓ Agente de IA inicializado correctamente")

✓ Agente de IA inicializado correctamente


C:\Users\martin\AppData\Local\Temp\ipykernel_15448\268745724.py:19: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(
C:\Users\martin\AppData\Local\Temp\ipykernel_15448\268745724.py:38: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


In [5]:
# Función para procesar mensajes del usuario
def chat_with_agent(message: str, history: list) -> Tuple[str, list]:
    """
    Procesa el mensaje del usuario y devuelve la respuesta del agente
    
    Args:
        message: Mensaje del usuario
        history: Historial de conversación (formato Gradio 6.2+)
    
    Returns:
        Tupla con (respuesta, historial actualizado)
    """
    if not message.strip():
        return "", history
    
    try:
        # Obtener respuesta del agente
        # NOTA: .predict() es el método legado. En versiones futuras considera usar .invoke() o .stream()
        response = conversation_chain.predict(input=message)
        
        # Actualizar el historial con formato compatible con Gradio 6.2+
        # El formato debe ser: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response})
        
        return "", history
    
    except Exception as e:
        # Manejo de errores mejorado: captura cualquier excepción y muestra mensaje informativo
        error_msg = f"Error al procesar el mensaje: {str(e)}"
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": error_msg})
        return "", history

# Función para obtener el directorio temporal (compatible con Gradio)
def get_temp_folder():
    """Obtiene la ruta del directorio temporal del sistema (compatible con Gradio)"""
    # Usar el directorio temporal del sistema que Gradio permite
    temp_dir = tempfile.gettempdir()
    return temp_dir

# Funciones auxiliares para exportación (eliminan código duplicado)
def extract_message_content(msg: dict) -> str:
    """Extrae el contenido de un mensaje como string, manejando listas y strings"""
    content_raw = msg.get("content", "")
    if isinstance(content_raw, list):
        return " ".join(str(item) for item in content_raw)
    return str(content_raw)

def generate_export_filepath(extension: str) -> str:
    """Genera la ruta del archivo de exportación con timestamp único"""
    temp_folder = get_temp_folder()
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"conversacion_agente_{timestamp}.{extension}"
    return os.path.join(temp_folder, filename)

# Funciones para exportar conversaciones
def export_to_txt(history: list) -> Optional[str]:
    """Exporta el historial a formato TXT y retorna la ruta del archivo"""
    if not history:
        return None
    
    # Generar contenido
    content = "=" * 60 + "\n"
    content += "CONVERSACIÓN CON AGENTE DE IA\n"
    content += f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    content += "=" * 60 + "\n\n"
    
    for msg in history:
        role = msg.get("role", "unknown")
        text = extract_message_content(msg)
        
        if role == "user":
            content += f"👤 Usuario:\n{text}\n\n"
        elif role == "assistant":
            content += f"🤖 Agente:\n{text}\n\n"
        content += "-" * 60 + "\n\n"
    
    # Guardar en directorio temporal (compatible con Gradio)
    filepath = generate_export_filepath("txt")
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return filepath

def export_to_markdown(history: list) -> Optional[str]:
    """Exporta el historial a formato Markdown y retorna la ruta del archivo"""
    if not history:
        return None
    
    # Generar contenido
    content = "# Conversación con Agente de IA\n\n"
    content += f"**Fecha:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "---\n\n"
    
    for msg in history:
        role = msg.get("role", "unknown")
        text = extract_message_content(msg)
        
        if role == "user":
            content += f"## 👤 Usuario\n\n{text}\n\n"
        elif role == "assistant":
            content += f"## 🤖 Agente\n\n{text}\n\n"
        content += "---\n\n"
    
    # Guardar en directorio temporal (compatible con Gradio)
    filepath = generate_export_filepath("md")
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return filepath

def export_to_pdf(history: list) -> Optional[str]:
    """Exporta el historial a formato PDF y devuelve la ruta del archivo"""
    if not history:
        return None
    
    # Crear archivo en directorio temporal (compatible con Gradio)
    temp_path = generate_export_filepath("pdf")
    
    # Crear documento PDF
    doc = SimpleDocTemplate(temp_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []
    
    # Estilos personalizados
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=16,
        textColor=HexColor('#000000'),
        spaceAfter=30,
        alignment=TA_LEFT
    )
    
    user_style = ParagraphStyle(
        'UserStyle',
        parent=styles['Normal'],
        fontSize=11,
        textColor=HexColor('#00008B'),
        spaceAfter=12,
        leftIndent=20
    )
    
    assistant_style = ParagraphStyle(
        'AssistantStyle',
        parent=styles['Normal'],
        fontSize=11,
        textColor=HexColor('#006400'),
        spaceAfter=12,
        leftIndent=20
    )
    
    # Título
    story.append(Paragraph("Conversación con Agente de IA", title_style))
    story.append(Paragraph(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
    story.append(Spacer(1, 0.3*inch))
    
    # Contenido
    for msg in history:
        role = msg.get("role", "unknown")
        text = extract_message_content(msg)
        # Reemplazar saltos de línea para HTML
        text = text.replace('\n', '<br/>')
        
        if role == "user":
            story.append(Paragraph(f"<b>👤 Usuario:</b>", styles['Normal']))
            story.append(Paragraph(text, user_style))
        elif role == "assistant":
            story.append(Paragraph(f"<b>🤖 Agente:</b>", styles['Normal']))
            story.append(Paragraph(text, assistant_style))
        
        story.append(Spacer(1, 0.2*inch))
    
    # Construir PDF
    doc.build(story)
    return temp_path

def export_to_docx(history: list) -> Optional[str]:
    """Exporta el historial a formato DOCX y devuelve la ruta del archivo"""
    if not history:
        return None
    
    # Crear documento
    doc = Document()
    
    # Título
    title = doc.add_heading('Conversación con Agente de IA', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT
    
    # Fecha
    date_para = doc.add_paragraph(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    date_para.runs[0].font.size = Pt(10)
    doc.add_paragraph()
    
    # Contenido
    for msg in history:
        role = msg.get("role", "unknown")
        text = extract_message_content(msg)
        
        if role == "user":
            heading = doc.add_heading('👤 Usuario', level=2)
            heading.runs[0].font.color.rgb = RGBColor(0, 0, 139)
            para = doc.add_paragraph(text)
            para.runs[0].font.size = Pt(11)
        elif role == "assistant":
            heading = doc.add_heading('🤖 Agente', level=2)
            heading.runs[0].font.color.rgb = RGBColor(0, 100, 0)
            para = doc.add_paragraph(text)
            para.runs[0].font.size = Pt(11)
        
        doc.add_paragraph()
    
    # Guardar en directorio temporal (compatible con Gradio)
    temp_path = generate_export_filepath("docx")
    doc.save(temp_path)
    
    return temp_path

def download_conversation(history: list, format_type: str) -> Optional[str]:
    """Genera y devuelve el archivo de conversación en el formato especificado"""
    if not history:
        return None
    
    # Diccionario de funciones de exportación (elimina código duplicado)
    export_functions = {
        "TXT": export_to_txt,
        "Markdown": export_to_markdown,
        "PDF": export_to_pdf,
        "DOCX": export_to_docx
    }
    
    if format_type not in export_functions:
        print(f"⚠ Formato no reconocido: {format_type}")
        return None
    
    try:
        filepath = export_functions[format_type](history)
        if filepath:
            print(f"✓ Archivo generado: {filepath}")
            print("  → Haz clic en el archivo para descargarlo")
        return filepath
    except Exception as e:
        error_msg = f"Error al exportar: {str(e)}"
        print(error_msg)
        return None

In [6]:
# Función para limpiar la conversación
def clear_conversation():
    """Reinicia la memoria de la conversación"""
    global conversation_chain
    conversation_chain = initialize_agent()
    return [], []  # Devolver lista vacía para el historial

# Crear la interfaz gráfica con Gradio
def create_chat_interface():
    """Crea la interfaz gráfica tipo ChatGPT"""
    
    with gr.Blocks(title="Agente IA - LangChain") as interface:
        gr.Markdown(
            """
            # 🤖 Agente de IA Conversacional
            ### Powered by LangChain + Groq (Llama 3.1)
            
            Chatea con el agente de IA. Mantiene el contexto de la conversación.
            """
        )
        
        chatbot = gr.Chatbot(
            label="Conversación",
            height=500,
            show_label=True,
            container=True
        )
        
        with gr.Row():
            msg = gr.Textbox(
                label="Tu mensaje",
                placeholder="Escribe tu mensaje aquí...",
                scale=4,
                container=False
            )
            submit_btn = gr.Button("Enviar", variant="primary", scale=1)
            clear_btn = gr.Button("Limpiar", variant="secondary", scale=1)
        
        # Sección de descarga
        with gr.Row():
            format_dropdown = gr.Dropdown(
                choices=["TXT", "Markdown", "PDF", "DOCX"],
                value="TXT",
                label="Formato de descarga",
                scale=2,
                interactive=True
            )
            download_btn = gr.Button("📥 Descargar Conversación", variant="secondary", scale=1)
        
        # Mostrar el formato seleccionado de forma clara
        format_info = gr.Markdown("**Formato seleccionado:** 📄 TXT (.txt)", elem_classes=["format-info"])
        download_file = gr.File(label="📥 Archivo descargable", visible=True, interactive=False)
        
        # Eventos
        msg.submit(chat_with_agent, [msg, chatbot], [msg, chatbot])
        submit_btn.click(chat_with_agent, [msg, chatbot], [msg, chatbot])
        clear_btn.click(clear_conversation, None, [chatbot, msg])
        
        # Función para actualizar la información del formato
        def update_format_info(format_type):
            """Actualiza el texto que muestra el formato seleccionado"""
            format_info_dict = {
                "TXT": "**Formato seleccionado:** 📄 TXT (.txt)",
                "Markdown": "**Formato seleccionado:** 📝 Markdown (.md)",
                "PDF": "**Formato seleccionado:** 📑 PDF (.pdf)",
                "DOCX": "**Formato seleccionado:** 📘 Word (.docx)"
            }
            return format_info_dict.get(format_type, f"**Formato seleccionado:** {format_type}")
        
        # Evento de descarga
        def handle_download(history, format_type):
            """Maneja la descarga y muestra el archivo"""
            if not history:
                print("⚠ No hay conversación para descargar")
                return None
            
            filepath = download_conversation(history, format_type)
            if filepath and os.path.exists(filepath):
                print(f"✓ Archivo {format_type} listo para descargar")
                return filepath
            else:
                print("✗ Error al generar el archivo")
                return None
        
        # Actualizar la información del formato cuando cambie la selección
        format_dropdown.change(
            update_format_info,
            inputs=format_dropdown,
            outputs=format_info
        )
        
        # Evento de descarga
        download_btn.click(
            handle_download,
            inputs=[chatbot, format_dropdown],
            outputs=download_file
        )
        
        gr.Markdown(
            """
            ---
            **Nota:** El agente mantiene el contexto de la conversación. 
            - Usa el botón "Limpiar" para reiniciar la conversación.
            - Selecciona un formato y usa el botón "Descargar Conversación" para generar el archivo.
            - Haz clic en el archivo generado para descargarlo a tu carpeta de descargas.
            """
        )
    
    return interface

# Crear la interfaz
interface = create_chat_interface()
print("✓ Interfaz gráfica creada")

✓ Interfaz gráfica creada


In [7]:
# Lanzar la interfaz gráfica
# Esto abrirá una ventana en tu navegador o mostrará un enlace local

# CSS personalizado (en Gradio 6.0+ se pasa a launch())
custom_css = """
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
.chat-message {
    padding: 10px;
    border-radius: 10px;
    margin: 5px 0;
}
"""

# Intentar encontrar un puerto disponible
import socket

def find_free_port(start_port=7860, max_attempts=10):
    """Encuentra un puerto libre empezando desde start_port"""
    for i in range(max_attempts):
        port = start_port + i
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            sock.bind(('127.0.0.1', port))
            sock.close()
            return port
        except OSError:
            continue
    return None  # Si no encuentra puerto, Gradio buscará uno automáticamente

# Buscar puerto libre
free_port = find_free_port(7860)
if free_port:
    print(f"✓ Usando puerto {free_port}")
else:
    print("⚠ No se encontró puerto libre en el rango, Gradio buscará uno automáticamente")
    free_port = None

interface.launch(
    share=False,  # Cambia a True si quieres un enlace público temporal
    server_name="127.0.0.1",  # Dirección local
    server_port=free_port,  # Puerto libre encontrado (o None para automático)
    inbrowser=True,  # Abre automáticamente en el navegador
    css=custom_css  # CSS personalizado
)

✓ Usando puerto 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


✓ Archivo generado: C:\Users\martin\AppData\Local\Temp\conversacion_agente_20260101_131505.txt
  → Haz clic en el archivo para descargarlo
✓ Archivo TXT listo para descargar
✓ Archivo generado: C:\Users\martin\AppData\Local\Temp\conversacion_agente_20260101_131512.pdf
  → Haz clic en el archivo para descargarlo
✓ Archivo PDF listo para descargar
✓ Archivo generado: C:\Users\martin\AppData\Local\Temp\conversacion_agente_20260101_131546.txt
  → Haz clic en el archivo para descargarlo
✓ Archivo TXT listo para descargar


## Uso del Agente

### Opción 1: Desde el Notebook
Ejecuta todas las celdas anteriores y luego ejecuta la última celda para lanzar la interfaz gráfica.

### Opción 2: Modo Conversación Directo (sin interfaz gráfica)
Si prefieres interactuar directamente desde el notebook, puedes usar la siguiente celda:

In [ ]:
# Modo conversación directo (sin interfaz gráfica)
# Descomenta y ejecuta esta celda para chatear directamente desde aquí

# while True:
#     user_input = input("\nTú: ")
#     if user_input.lower() in ['salir', 'exit', 'quit', 'adios']:
#         print("¡Hasta luego!")
#         break
#     response = conversation_chain.predict(input=user_input)
#     print(f"\nAgente: {response}")

## Personalización

### Cambiar el Modelo
Puedes cambiar el modelo en la función `initialize_agent()`:
- `llama-3.1-8b-instant`: Modelo rápido y eficiente (recomendado, actualmente disponible)
- `llama-3.3-70b-versatile`: Modelo potente (verificar disponibilidad)
- `mixtral-8x7b-32768`: Modelo con contexto largo
- ⚠️ `llama-3.1-70b-versatile`: **DESCONTINUADO** - no usar

**Nota:** Algunos modelos pueden estar descontinuados. Consulta https://console.groq.com/docs/models para ver los modelos disponibles.

### Ajustar la Temperatura
- `temperature=0.0`: Respuestas más deterministas y precisas
- `temperature=0.7`: Balance entre creatividad y precisión (recomendado)
- `temperature=1.0`: Respuestas más creativas y variadas

### Cambiar el Prompt del Sistema
Modifica el `SystemMessagePromptTemplate` en `initialize_agent()` para personalizar el comportamiento del agente.